# 워밍업 — 한 번은 맞히는데 열 번은 못 맞힌다

웹 챕터(워밍업)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산해 봅니다.
웹의 실험실(ChainLab)과 **비트 단위로 같은 난수열**(mulberry32)을 쓰므로 결과가 정확히 일치해요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('sample')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 실험 준비: 신의 시점

**걸음 정확도** p(한 단계를 맞힐 확률)를 미리 정해둔 AI 일꾼에게, 단계 수 n짜리 심부름을 맡깁니다.
한 단계라도 틀리면 일 전체가 실패합니다. 잘못된 표가 들어간 보고서처럼요.

웹과 같은 난수 생성기(mulberry32)를 파이썬으로 옮겨 씁니다. 시드 = `p% × 100000 + n × 100 + 회차`.

In [ ]:
def mulberry32(seed):
    """웹(ChainLab.vue)과 비트 단위로 동일한 결정적 난수 생성기."""
    a = seed & 0xFFFFFFFF
    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & 0xFFFFFFFF
        t = a
        t = ((t ^ (t >> 15)) * (t | 1)) & 0xFFFFFFFF
        inner = ((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF
        t = (t ^ ((t + inner) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    return rand

def run_task(p_pct, n, rnd):
    """심부름 1회 — 처음 실패한 단계 번호(1부터), 끝까지 성공하면 0."""
    rand = mulberry32(p_pct * 100000 + n * 100 + rnd)
    for i in range(1, n + 1):
        if rand() >= p_pct / 100:
            return i
    return 0

# 걸음 정확도 90%, 10단계 일을 20번 맡기면 —
fails = [run_task(90, 10, r) for r in range(1, 21)]
wins = sum(1 for f in fails if f == 0)
print('회차별 결과(0=성공, 숫자=실패한 단계):', fails)
print(f'성공: {wins}/20')

# 웹 실험실과 완전히 같은 결과인지 자가검증
assert fails == [0, 0, 9, 0, 7, 6, 8, 9, 7, 7, 1, 0, 0, 3, 6, 0, 0, 0, 0, 1], '웹과 난수열이 다릅니다!'
assert wins == 9
# 단계 수별 20회 성공 횟수(웹 ③ 요약과 동일)
wins_by_n = [sum(1 for r in range(1, 21) if run_task(90, n, r) == 0) for n in [1, 3, 5, 10, 20]]
print('단계 수 [1,3,5,10,20]별 성공/20 :', wins_by_n)
assert wins_by_n == [19, 16, 11, 9, 1]
print('✅ 웹 실험실과 비트 단위 일치')

## 2. 이론: 일의 성공률은 걸음의 곱

각 단계가 **독립**이면(3부 3장의 곱셈 규칙), n단계를 전부 성공할 확률은

$$P(\text{일 성공}) = p \times p \times \cdots \times p = p^n$$

걸음 90%가 10단계 일에서는 $0.9^{10} \approx 35\%$. 실험의 9/20(45%)이 이 이론값 주위에서 출렁였던 겁니다
(20회 요약도 표본이라서요. 기대값은 20 × 0.349 ≈ 7번).

In [ ]:
import math

p = 0.9
print(f'0.9^10 = {p**10:.4f}  (약 {p**10*100:.0f}%)')
assert abs(p**10 - 0.3487) < 5e-4
assert round(p**10 * 100) == 35

# 걸음 정확도별 '성공률이 절반 아래로 떨어지는 단계 수'
for pct in [80, 90, 95, 99]:
    q = pct / 100
    half = math.ceil(math.log(0.5) / math.log(q))
    print(f'걸음 {pct}% → {half}단계부터 성공률 절반 미만 (10단계 일은 {q**10*100:.1f}%)')
assert math.ceil(math.log(0.5) / math.log(0.9)) == 7
assert abs(0.99**10 - 0.9044) < 5e-4  # 걸음 +9%p → 일 +55%p 의 지렛대
assert abs(0.95**10 - 0.5987) < 5e-4  # 광고의 95%는 10단계 업무에선 60%
print('✅ 본문 수치 자가검증 통과')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ns = range(1, 21)
fig, ax = plt.subplots(figsize=(8, 4.5))
for pct, color in [(80, '#f43f5e'), (90, '#6366f1'), (95, '#f59e0b'), (99, '#10b981')]:
    q = pct / 100
    ax.plot(list(ns), [q**n * 100 for n in ns], marker='o', ms=3, color=color, label=f'걸음 {pct}%')
    ax.annotate(f'{q**10*100:.0f}%', (10, q**10 * 100), textcoords='offset points', xytext=(6, 4), color=color, fontsize=9)
ax.axhline(50, ls='--', lw=1, color='#94a3b8')
ax.axvline(10, ls=':', lw=1, color='#94a3b8')
ax.set_xlabel('단계 수 n')
ax.set_ylabel('일 전체의 성공률 (%)')
ax.set_title('일의 성공률 = 걸음의 곱 p^n: 사슬이 길수록 지수적으로 하락')
ax.set_xticks([1, 3, 5, 10, 15, 20])
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 오늘의 발견

- **일의 성공률 = 걸음의 곱** ($p^n$): 걸음 90%도 10단계면 35%, 20단계면 12%.
- 걸음 정확도의 작은 개선(90→99%)이 긴 일에서는 큰 차이(35→90%)가 된다. 지수의 지렛대.
- 실험의 성공 횟수(9/20)도 표본이라 이론값(35%) 주위에서 출렁인다. 측정의 ±는 11장에서 다시 쓰인다.
- 실제 에이전트의 단계는 완전 독립이 아니다(오류 전파). 독립 가정은 출발점 모형(5·9장에서 균열을 다룸).

다음 장(1장 큰 그림)에서 AI 일꾼의 몸(상태·행동·도구·루프)을 해부합니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**